# Agentic Workflows

## Pattern For Highly Autonomous Agents - Planning Workflows

An agentic system that generates a short research report through planning, external tool usage, and feedback integration. Workflow will involve:


### 👥 Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.


### 🧰 Available Tools
* `arxiv_search_tool()`
* `tavily_search_tool()`
* `wikipedia_search_tool()`

Each tool can be called as part of an agent workflow.


Function `generate_research_report(prompt: str) -> dict` that orchestrates the full workflow:

1. The **planner agent** creates a research plan.
2. The **research agent** fetches external content based on the plan.
3. The **planner** writes a first draft.
4. The **editor agent** reflects on the draft and provides feedback.
5. The **planner** revises the draft using the feedback.

Use **tool calls** and **reasoning steps** where appropriate. No hard-code queries, the agents generate them dynamically.

# Import Libraries and load environment

In [1]:
import sys 
from pathlib import Path

parent_dir = Path().resolve().parent
sys.path.append(str(parent_dir))

from research import research_tools
from datetime import datetime
from IPython.display import Markdown, display
from aisuite import Client
from dotenv import load_dotenv

import re
import json


load_dotenv()

client = Client()

### Planner agent

Generates a **step-by-step research plan** as a Python list of strings

Each step must:
- Be executable by one of the available agents (research_agent, writer_agent, editor_agent)
-,Be clearly written and atomic
- Avoid unrelated tasks like file handling or installing packages
- End with a final step that **generates a Markdown document** with the research report


**Parameters**
- Model: "openai:o4-mini"
- Temperature: 1.0

In [2]:
def planner_agent(topic: str, model: str = "openai:4o-mini") -> list[str]:
  """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
      topic (str): Research topic to investigate.
      model (str): Language model to use.

    Returns:
      List[str]: A list of executable step strings.
  """

  prompt = f"""
  You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

  Available agents:
  - A research agent who can search web, wikipedia and arXiv.
  - A writer agent who can draft research summaries.
  - A editor agent who can reflect and revise the drafts.

  Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
  Each step should be atomic, executable and must rely only on the capabilities of the above agent.

  Do NOT include markdown code fences (```), explanations, or any text outside the Python list.
  Do not include irrelevant tasks like 'Create csv', ' Set up a repo', 'install packages'. etc.
  Do include research related tasks (e.g., search summarize, draft, revise)
  Do assume tool use is available
  Do not include explanation text - return only the Python list
  The final step should be to generate a Markdown document containing the complete research report.

  Topis: "{topic}"
  """

  response = client.chat.completions.create(
    model=model,
    messages=[{ "role": "user", "content": prompt}],
    temperature=1
  )

  steps = eval(response.choices[0].message.content.strip())

  return steps

In [3]:
steps = planner_agent("The ensemble Kalman filter for time series forecasting", "openai:gpt-4o")

### Research Agent

Research_agent(task: str) -> str that executes a research task using tools like arXiv, Tavily, and Wikipedia

In [4]:
def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False) -> str:
  print("==================================")
  print("🔍 Research Agent")
  print("==================================")
  
  prompt = f"""
    You are a research assistant with access to the following tools:
    - arxiv_tool: for finding academic papers
    - tavily_tool: for general web search
    - wikipedia_tool: for encyclopedic knowledge

    Task:
    {task}

    Today is {datetime.now().strftime('%Y-%m-%d')}.
  """

  messages = [{ "role": "user", "content": prompt.strip() }]
  tools = [research_tools.arxiv_search, research_tools.tavily_search, research_tools.wikipedia_search]

  try:
    response = client.chat.completions.create(
      model=model,
      messages=messages,
      tools=tools,
      tool_choice="auto",
      max_turns=12
    )
    content = response.choices[0].message.content
    print("✅ Output:\n", content)
    return (content, messages) if return_messages else content
  
  except Exception as e:
    print("❌ Error:", e)
    return f"[Model Error: {str(e)}]"

### Writer agent

writer_agent(task: str) -> str that handles writing tasks like drafting sections or summarizing content.

In [5]:
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str:
  print("==================================")
  print("✍️ Writer Agent")
  print("==================================")
  messages = [
        {"role": "system", "content": "You are a writing agent specialized in generating well-structured academic or technical content."},
        {"role": "user", "content": task}
    ]

  response = client.chat.completions.create(
      model=model,
      messages=messages,
      temperature=1.0
  )

  return response.choices[0].message.content

### Editor agent
editor_agent(task: str) -> str that performs editorial tasks like revision and reflection.

In [6]:
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
  print("==================================")
  print("🧠 Editor Agent")
  print("==================================")
  messages = [
      {"role": "system", "content": "You are an editor agent. Your job is to reflect on, critique, or improve existing drafts."},
      {"role": "user", "content": task}
  ]

  response = client.chat.completions.create(
      model=model,
      messages=messages,
      temperature=0.7
  )

  return response.choices[0].message.content

### Executor agent

executor_agent(plan_steps: List[str]) routes each task to the correct sub-agent (research_agent, writer_agent, or editor_agent) and maintains a history of all steps.

In [7]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [8]:
def executor_agent(plan_steps: list[str], model: str = "openai:gpt-4o"):
  history = []

  print("==================================")
  print("🎯 Executor Agent")
  print("==================================")

  for _, step in enumerate(plan_steps):
    agent_decision_prompt = f"""
      You are an execution manager for a multi-agent research team.

      Given the following instruction, identify which agent should perform it and extract the clean task.

      Return only a valid JSON object with two keys:
      - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
      - "task": a string with the instruction that the agent should follow

      Only respond with a valid JSON object. Do not include explanations or markdown formatting.

      Instruction: "{step}"
    """
    response = client.chat.completions.create(
      model=model,
      messages=[{"role": "user", "content": agent_decision_prompt}],
      temperature=0,
    )

    raw_content = response.choices[0].message.content
    cleaned_json = clean_json_block(raw_content)
    agent_info = json.loads(cleaned_json)

    agent_name = agent_info["agent"]
    task = agent_info["task"]

    context = "\n".join([
      f"Step {j+1} executed by {a}:\n{r}" 
      for j, (_, a, r) in enumerate(history)
    ])
    enriched_task = f"""You are {agent_name}.
      Here is the context of what has been done so far:
      {context}

      Your next task is:
      {task}
    """

    print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

    if agent_name in agent_registry:
      output = agent_registry[agent_name](enriched_task)
      history.append((step, agent_name, output))
    else:
      output = f"⚠️ Unknown agent: {agent_name}"
      history.append((step, agent_name, output))

    print(f"✅ Output:\n{output}")

  return history



In [9]:
executor_history = executor_agent(steps)

🎯 Executor Agent

🛠️ Executing with agent: `research_agent` on task: Search web for 'ensemble Kalman filter time series forecasting overview'
🔍 Research Agent
✅ Output:
 I've retrieved some results on the topic of "ensemble Kalman filter time series forecasting overview." Here are the top 5 findings:

1. **Improving Numerical Forecast Accuracy with Ensemble Kalman Filter**  
   This study proposes a new hybrid data assimilation scheme combining chaos theory with the ensemble Kalman filter to enhance prediction capability. It has been applied in a simulated real-time forecast of a river model.
   [Read more here](https://www.sciencedirect.com/science/article/abs/pii/S0022169414001905)

2. **Ensemble Kalman Filtering with One-Step-Ahead Smoothing**  
   This article discusses a method where the smoothed state is integrated forward in time, incorporating observations to update forecasts through different types of analyses and forecasts.
   [Read more here](https://journals.ametsoc.org/vie

In [10]:
md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

# Ensemble Kalman Filter in Time Series Forecasting: An Overview

## Introduction to the Ensemble Kalman Filter

The Ensemble Kalman Filter (EnKF) is a pivotal tool in statistical data assimilation and time series forecasting, particularly for large-scale dynamical systems. Emerging as an extension of the traditional Kalman Filter, the EnKF addresses computational challenges in high-dimensional state spaces, common in geophysical and atmospheric sciences [1].

The EnKF uses a Monte Carlo approach to approximate solutions to the Bayesian update problem. Instead of directly computing the error covariance matrix, it employs an ensemble of system states that evolve with the system's dynamics. This ensemble-based method effectively handles nonlinearity and model errors, making it attractive for various applications, from weather forecasting to economic modeling [2].

A key feature of the EnKF is its ability to integrate observational data in real-time. By continuously updating the ensemble of states, it refines predictions and systematically reduces the uncertainty inherent in predictions based on incomplete or noisy data [3].

Recent advancements have seen the EnKF extend its capabilities through hybrid data assimilation schemes, chaos theory integration, and machine learning models, demonstrating its versatility and robustness. As such, the EnKF remains essential for researchers and practitioners aiming to improve forecast precision and optimize predictions in complex, dynamic environments [4].

## Application of the Ensemble Kalman Filter in Time Series Forecasting

The EnKF enhances time series forecasting across various scientific and practical applications through its ensemble approach. Below is a detailed summary of research findings on its application:

1. **Integration with Existing Models**: The EnKF improves forecasting by integrating with existing statistical and dynamical models, such as combining with chaos theory for real-time river modeling. This integration is effective in managing nonlinear dynamics and stochastic variations [5].

2. **Energy and Environmental Forecasting**: In energy forecasting, the EnKF enhances predictions of energy demands and generation, optimizing grid operations. In environmental studies, it predicts atmospheric phenomena like storm surges and dust storms by assimilating real-time observational data [6][7].

3. **Improving Numerical Forecast Models**: The EnKF refines system state estimations, as seen with the Valid Time Shifting-EnKF, which enhances prediction accuracy for dust storms by adapting to new datasets [8].

4. **Machine Learning Integration**: The EnKF is integrated with machine learning techniques, such as in the U-Net Kalman Filter (UNetKF), combining pattern recognition with sequential prediction to enhance data assimilation and state estimation [9].

5. **Nonlinear and Non-Gaussian Models**: The EnKF can be applied to nonlinear and non-Gaussian state-space models, achieving convergence and improving forecast reliability despite potential nonlinearity and uncertainty [10].

6. **Comparative Assessments**: The EnKF is evaluated against other filtering methods, showcasing its computational efficiency and adaptability in assimilating diverse datasets. Innovations like integration with Gaussian process state-space models have expanded its applicability and accuracy [11].

Overall, the EnKF's adaptability makes it central to advancing time series forecasting, supported by hybrid approaches and integrations.

## Benefits and Limitations of Using the Ensemble Kalman Filter

The EnKF offers significant advantages for time series forecasting but also presents limitations:

### Benefits

1. **Scalability and Efficiency**: The EnKF is scalable and suitable for high-dimensional data, offering computational efficiency without needing expensive error covariance matrix computations [12].

2. **Handling Nonlinearity and Uncertainty**: It manages nonlinear dynamics and uncertainties with robust estimates in complex environments [13].

3. **Real-Time Data Integration**: The EnKF incorporates new data in real-time, valuable for fields requiring real-time analysis [14].

4. **Flexibility**: It integrates with other models and frameworks, enhancing predictive accuracy [15].

5. **Reduced Computational Complexity**: The EnKF requires fewer ensemble members than particle filters, reducing computational burden [16].

### Limitations

1. **Gaussian Assumptions**: The EnKF assumes Gaussian distributions, which may not hold in all systems, leading to biased estimates [17].

2. **Dependency on Ensemble Size**: Performance relies on ensemble size, balancing accuracy and computational cost [18].

3. **Sensitivity to Model Errors**: The EnKF is sensitive to model biases, which can lead to inaccurate predictions [19].

4. **Initialization Challenges**: Initial conditions influence accuracy, necessitating careful calibration [20].

5. **Complex Implementation**: Implementing the EnKF can be complex, limiting accessibility for some practitioners [21].

## Case Studies and Applications

1. **Dust Storm Forecasting**: The VTS-EnKF combines stochastic EnKF with time-shifting for accurate dust storm forecasting [8].

2. **Model-Free Filtering**: The Kalman-Takens filter isolates predictable components in phenomena like El Niño without explicit models [22].

3. **Spatio-Temporal Phenomena**: Resampling techniques improve data assimilation for atmospheric sciences [23].

4. **Storm Surge Assimilation**: Comparative studies assess EnKF variants’ efficacy in storm surge data assimilation [24].

5. **Machine Learning Integration**: The UNetKF blends machine learning with the EnKF for enhanced state estimation [9].

## Conclusion and Future Research Directions

The EnKF remains crucial in time series forecasting, enhancing accuracy and reliability. Future research should explore:

1. **Machine Learning Synergies**: Deepening integration with machine learning for precise predictions in complex contexts.

2. **Hybrid Models**: Developing models that address Gaussian limitations.

3. **Efficiency Improvements**: Enhancing computational efficiency for large datasets.

4. **Environmental Robustness**: Improving adaptability under varying conditions.

5. **Cross-Disciplinary Applications**: Expanding into new fields like social networks.

The EnKF's evolution will continue to drive advancements in predictive analytics, offering innovative solutions to complex forecasting challenges across disciplines.

## References
1. Wikipedia article on Ensemble Kalman Filter
2. Improving Numerical Forecast Accuracy with Ensemble Kalman Filter
3. Ensemble Kalman Filtering with One-Step-Ahead Smoothing
4. A Brief Tutorial on the Ensemble Kalman Filter
5. A Multi-Model Ensemble Kalman Filter for Data Assimilation and Forecasting
6. Application of Ensemble Kalman Filter in Forecasting the Electricity Grid Carbon Factor
7. The Ensemble Kalman Filter: Theoretical Formulation and Practical Implementation
8. Valid Time Shifting Ensemble Kalman Filter (VTS-EnKF) for Dust Storm Forecasting
9. U‐Net Kalman Filter (UNetKF): An Example of Machine Learning for Data Assimilation
10. Understanding the Ensemble Kalman Filter - UMD Math Department
11. Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference
12. Resampling the Ensemble Kalman Filter
13. The Geometric Unscented Kalman Filter
14. LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting
15. SCINet: Time Series Modeling and Forecasting with Sample Convolution and Interaction
16. Ensemble Kalman Filtering without a Model
17. A Comparison of Ensemble Kalman Filters for Storm Surge Assimilation
18. Probabilistic Hierarchical Forecasting with Deep Poisson Mixtures
19. The Kalman Filter and State Estimation
20. Applications of the Ensemble Kalman Filter
21. Challenges in Implementing the Ensemble Kalman Filter
22. Ensemble Kalman Filtering without a Model
23. Resampling the Ensemble Kalman Filter
24. A Comparison of Ensemble Kalman Filters for Storm Surge Assimilation

This document offers a comprehensive overview of the Ensemble Kalman Filter's role in time series forecasting, citing various studies and literature to support the discussion.